# Análisis y visualización de datos con Python
# 6. Limpieza de datos

    - a) Selección de observaciones y variables
    - b) Tipos de datos
    - c) Valores faltantes
    - d) Limpieza de texto
    - e) Limpieza de categóricos
    - f) Limpieza de números
    - g) Limpieza de fechas
    - h) Ordenar y guardar datos
    - i) Resumen

---

La limpieza de datos es el proceso de identificar, corregir o eliminar datos incorrectos, incompletos, irrelevantes o duplicados en un conjunto de datos. La limpieza de datos es importante para garantizar la precisión y la calidad de los datos y evitar errores en el análisis posterior. Algunas técnicas comunes de limpieza de datos incluyen la eliminación de valores atípicos, la eliminación de valores faltantes, la corrección de errores tipográficos y la eliminación de duplicados.

Empezaremos cargando los datos.

In [1]:
from numpy import nan
import pandas as pd

filename = 'data_raw/CNB_DOB_BPGS_Respuesta_Solicitud 332163723000249.xlsx'
df = pd.read_excel(filename, sheet_name="HBO", index_col='ID' )
df.head()

,Numero_progresivo_transcrito,Nombre_completo_transcrito,Primer_apellido,Segundo_apellido,Nombres_propios,Fecha_transcrito,Fecha_estandar,Expediente_SEMEFO_transcrito,Procedencia_transcrito,Procedencia_estandar,...,Diagnostico_estandar,Diagnostico_extendido,Sexo,Edad_transcrito,Tipo_restos,Bitacora_ingresos,Pagina_PDF,Foja_transcrito,Observaciones,Conocido_desconocido
ID,,,,,,,,,,,,,,,,,,,,,
BO_1968_00001,S-D,acosta ortega teresa,acosta,ortega,teresa,1968-01-03 00:00:00,1968-01-03,37,S-D,S-D,...,S-D,sin datos,Femenino,S-D,Cadáver,semefo_df_bo_1968,2,1,NaN,conocido
BO_1968_00002,S-D,avila de cuestas catalina,avila,de cuestas,catalina,1968-01-05 00:00:00,1968-01-05,58,S-D,S-D,...,S-D,sin datos,Femenino,S-D,Cadáver,semefo_df_bo_1968,2,1,NaN,conocido
BO_1968_00003,S-D,arzate paredes juan,arzate,paredes,juan,1968-01-07 00:00:00,1968-01-07,83,S-D,S-D,...,S-D,sin datos,Masculino,S-D,Cadáver,semefo_df_bo_1968,2,1,NaN,conocido
BO_1968_00004,S-D,alvarez martinez isaac,alvarez,martinez,isaac,1968-01-07 00:00:00,1968-01-07,86,S-D,S-D,...,S-D,sin datos,Masculino,S-D,Cadáver,semefo_df_bo_1968,2,1,NaN,conocido
BO_1968_00005,S-D,arellano viuda de campos ma.,arellano,viuda de campos,ma.,1968-01-07 00:00:00,1968-01-07,88,S-D,S-D,...,S-D,sin datos,Femenino,S-D,Cadáver,semefo_df_bo_1968,2,1,NaN,conocido


Una consideración importante es si se modificara la columna original o se generará una columna nueva con los datos limpios. Qué opción se escoge depende de varios factores, cómo el tamaño de la tabla o si nos interesa conservar la información original. En este tutorial crearemos columnas nuevas para poder contrastar los cambios y al final quitaremos las que no nos interesen.


En este caso haremos una serie de correciones:
* Quitar columnas
* Cambiar tipos de datos
* Modificar campos de texto
* Llenar datos faltantes
* Estandarizar catálogos
* Eliminar datos fuera de rango
* Generar columnas con datos derivados
* Guardar datos

## 6.a Eliminación de observaciones y variables

Es posible que no todos los datos contenidos en el conjunto de datos sean de interés. Eliminarlos facilita el análisis y eficientiza el uso de memoria.

Podemos eliminar filas o columnas si:
* Baja variabilidad, por ejemplo una columna donde todos los valores son iguales
* Información repetida, por ejemplo filas idénticas
* Existe una gran cantidad de datos faltantes
* Información no relevante para el análisis.

Podemos quitar las columnas usando el comando `.drop(axis=1)`. Si lo que queremos es quitar filas, se puede hacer de una manera similar, poniendo los nombres de las filas entre corchetes y usando el parámetro `axis=0`.

Mucha de esta información se puede obtener del diccionario de datos y el profile.
* `Numero_progresivo_transcrito`

Por ejemplo, en el subconjunto que estamos usando las columnas 'Estado_origen', 'Municipio_origen' y 'Rdoc' tienen un solo valor, por lo que las eliminaremos

Para guardar estas modificaciones debemos de actualizar el dataframe.

In [11]:
df['Observaciones'].unique()

array([nan, 'expediente_semefo subrayado de rojo',
       'salto hasta la pagina _pdf 11 y foja 7 porque la pagina pdf 10 (foja 5) esta vacia.',
       ..., 'punto rojo en expediente_semefo. asterisco rojo en fecha.',
       'punto rojo en expediente_semefo y raya roja en procedencia. no copias rl',
       'no se recibio necropsia. es una osamenta.'], dtype=object)

In [ ]:
col_drop = ['Estado_origen', 'Municipio_origen', 'Rdoc']
df = df.drop( col_drop, axis=1)
df.head()

Para determinar filas duplicadas usamos el comando `duplicated()`.

In [ ]:
df[ df.duplicated() ]

Podemos ver que hay un número de filas que tienen datos duplicados, es decir, que todos los datos se parecen. 
En este caso todos son `Desconocidos` y tienen identificadores distintos, por lo que lo mas probable es que no sean duplicados de verdad, sino un caso de registros indistinguibles por la falta de información.

En caso de querer quitarlos se pueden quitar usando el comando `drop_duplicates()`, el cual incluye opciones para determinar cual de los duplicados conservar.


## 6.b Tipos de datos

Cómo vimos anteriormente hay varios tipos de datos como: texto, númerico, fecha, etc.
De manera automática pandas infiere el tipo de datos al leer el archivo, de una manera muy similar a como Excel interpreta si un dato es texto o numérico.

Los tipos de datos de pandas son:
* `object`: texto o mezcla de varios tipos de datos, este es el tipo de dato por defecto
* `category`: información categórica, similar a Factors en R
* `int64`: números enteros
* `float64`: números con punto decimal
* `bool`: valores verdadero o falso
* `datetime64`: fechas
* `timedelta[ns]`: diferencias de tiempo

Podemos ver los tipos de datos inferidos usando la opción `.dtypes`.

In [2]:
df.dtypes

Numero_progresivo_transcrito            object
Nombre_completo_transcrito              object
Primer_apellido                         object
Segundo_apellido                        object
Nombres_propios                         object
Fecha_transcrito                        object
Fecha_estandar                  datetime64[ns]
Expediente_SEMEFO_transcrito            object
Procedencia_transcrito                  object
Procedencia_estandar                    object
Procedencia_direccion                   object
Procedencia_alcaldia                    object
Numero_acta_transcrito                  object
Procedencia_acta                        object
Diagnostico_transcrito                  object
Diagnostico_estandar                    object
Diagnostico_extendido                   object
Sexo                                    object
Edad_transcrito                         object
Tipo_restos                             object
Bitacora_ingresos                       object
Pagina_PDF   

Dependiendo del tipo de dato se pueden hacer diferentes operaciones y hay tipos de limpieza específica.

Cómo vimos en la definición de datos ordenados o tidy data idealmente cada tabla tendrá solo datos de un tipo. Esto es para facilitar las operaciones. Si esto no es posible es importante que una columna o variable tenga datos de un solo tipo, ya que esto puede generar errores.

Por ejemplo, si tratamos de obtener el promedio de las edades nos genera un error de `ValueError`, ya que hay valores de texto `str` y número `int` mezclados, además de datos faltantes.

In [3]:
df['Edad_transcrito'].unique()

array(['S-D', ' S-D', '6 dias', '16 días', '30 dias', '4 meses',
       '3 meses', '14 dias', '18 dias', '3 semanas', '2 meses', '63',
       '17', '69', '40', '68', '50', '42', '66', '55', '60', '30', '48',
       '39', '35', '37', '56', '62', '27', '4', '41', '25', '72', '16',
       '65', '26', '8', '20', '3m', '3a', '28', '58', '23', '22', '4a',
       '75', '19m', '24', '14m', '54', '19', '34', '80', '13', '36', '45',
       '2', '61', '12', '70', '49', '1', '95', '3', '33', '32', '10 días',
       '14', '57', '85', '16M', '21', '5', '74', '7', '15', '6', '86',
       '11', '18m', '47', '29', '52', '81', '84', '18', '38', '43', '53',
       '1.5', '9m', '2m', '9', '44', '64', '73', '82', '83', '6m', '4m',
       '51', '8 días', '77', '10', '31', '46', '8 meses', '59',
       '18 meses', '25 dias', '20 meses', '6 meses', '79', '9 meses',
       '71', '67', '78', '1 mes', '90', '76', '5 meses', '2 meses ',
       '20 días', '14 meses', '98', '11 días', '3meses', '93', '15 días',
   

Existen varias formas de fijar el tipo de datos, una de ellas es usar la opción `dtype` al leer el dataframe. Esta opción puede recibir un solo tipo de datos o un diccionarió de tipos de datos.

Por ejemplo, podemos leer toda la tabla cómo texto usando `dtype='str'`, lo cual obligará a ciertos tipos de datos como fechas o números a aparecer cómo texto. Hacer esto es una buena opción con los profiles iniciales cuando hay columnas que no se leen correctamente.

También se puede pasar un diccionario marcando el tipo de datos de cada columna específicamente. Sin embargo, si los datos no están limpios se pueden generar errores de lectura. Esta opción es ideal ya que los datos están limpios.

Otra opción es convertir el tipo de dato columna por columna usando comandos como `.as_type` `.to_numeric` o `.to_datetime`.

![Función as_type](./images/pandas_astype.png)

Explicaremos esto con más detalle adelante.

In [ ]:
df = pd.read_excel(file_mfc, sheet_name="MFC", nrows=1000, index_col='ID', dtype='str' )
df.head()

Los tipos de datos aparecen cómo object

In [ ]:
df.dtypes

Los números de `Edad` aparecen entre comillas ya que son texto.

In [ ]:
df['Edad'].unique()

## 6.c Valores faltantes

Un valor faltante es un valor que falta o está incompleto en un conjunto de datos. A menudo, los valores faltantes se deben a errores de entrada de datos, fallas en la recolección de datos o problemas técnicos. Los valores faltantes pueden aparecer como espacios en blanco, ceros, letras u otros caracteres. La presencia de valores faltantes en un conjunto de datos puede afectar la precisión y la calidad del análisis de los datos, por lo que es importante identificarlos y manejarlos adecuadamente durante la limpieza y el preprocesamiento de los datos.

Otra razón para quitar datos es por que hay una gran cantidad de faltantes. El comando `dropna()` sirve para quitar estas filas (`axis=0`) o columnas (`axis=1`).
Existen varias opciones:

* `how='any'` quita las filas o columnas donde hay al menos un nan, este parametró es el default
* `how='all'` quita las filas o columnas donde todos son nan
* `thresh=numero` quita las filas o columnas con más de cierto número de nans, esto es útil para detectar automáticamente columnas donde faltan muchos datos

Quitemos las columnas donde todos los valores son nan, en este caso 'Datos alternativos':

In [ ]:
df = df.dropna( axis=1, how='all' )
df.head()

Otra situación es cuando la columna o fila solo tiene algunos faltantes, en este caso es necesario tomar decisiones dependiendo de la situación.

Por ejemplo, la columna de 'Fecha_exhumación' tiene 99.9% de datos faltantes. 
Esta columna es un caso interesante, ya que contiene información del unico caso de exhumación de nuestro subconjunto de datos. Podríamos quitarla dada la cantidad de datos faltantes, pero antes de eso es importante ver la fila asociada.

In [ ]:
df[ df['Fecha_exhumación'].notna() ]

En este caso es importante considerar que tipo de análisis nos interesa hacer y que información es necesaria.

Por ejemplo, si nos interesa las exhumaciones debemos de conservar la columna. Si nos interesan solo los cadaveres presentes en la fosa común se recomienda quitar la columna y la fila, ya que el cadaver ha sido removido. Sin embargo, si nos interesan las inhumanciones se debé de conservar la fila aunque se quite la columna, ya que el registro es tanto de una inhumación como de una exhumación. 
En general, es importante revisar los datos y tener claras las preguntas para tomar decisiones sobre la limpieza y análisis.

En este caso quitaremos las columnas con demasiados nan's, por ejemplo, dejaremos aquellas que tengan al menos 50 valores validos, lo cual debe de quitar 'Fecha_exhumación'

In [ ]:
df = df.dropna(thresh=50, axis=1)
df.head()

Para determinar si debemos de quitar filas por nan en primer lugar hay que análizar la cantidad de faltantes y su distribución.

**Nota** Explicación del comando
```
df.isna()         <-- determinar si las celdas son na
  .sum(axis=1)    <-- sumar los faltantes por fila
  .value_counts() <-- contar cuantas veces aparece cada número
  .sort_index()   <-- ordenar por el indice, que en este caso es el número de faltantes
```


In [ ]:
df.isna().sum(axis=1).value_counts().sort_index()

Cómo podemos ver hay 285 filas donde faltan 4 valores, lo cual es poco menos de la mitad. Revisemos estas filas.

In [ ]:
df[  df.isna().sum(axis=1)>=4  ]

In [ ]:
df[ df.isna().sum(axis=1)>=4 ]

En este caso son desconocidos donde no se sabe la edad, por lo que no es necesario quitarlas.

### Valores especiales

Otra opción es llenar los faltantes con un valor específico o especial. Por ejemplo, veamos la columna `Fecha_defunción`.
En esta columna hay varios tipos de valores faltantes, los cuales son representados no por `nan`, sino por 'S/D' y '??'

Esta es una forma de representar incertidumbre en los datos. Los datos faltantes pueden provenir de varias fuentes. Por ejemplo, en este caso el MFC incluye transcripciones de los registros de las fosas comunes del país. Es diferente que un dato no este en la fuente original a que no haya sido capturado al MFC. 

Existen varias formas de tratar estos datos faltantes, por ejemplo en este caso no se tiene la hora de defunción, por lo que por convención se representa la hora cómo 00:00:00, esto no significa que la persona haya muerto a esa hora, sino que es un valor default. El llenar los datos con valores default puede sesgar la distribución de los datos.

In [ ]:
df['Fecha_defunción'].unique() [0:25]

Otro ejemplo es lo que sucede con los nombres. En este caso tenemos varios valores especiales, ya que hay nombres de desconocidos que se representan con un espacio '  ', con `nan` o con el texto 'Nombre de particular que se encuentra con vida, se clasifica como confidencial con fundamento en el artículo 116 de la LGTAIP.  '. Cada uno de estos representa diferentes tipos de falta de información y amerita un tratamiento diferente.

In [ ]:
df['Nombre completo'].value_counts(dropna=False)[0:10].index

In [ ]:
df['Nombre(s)'].value_counts(dropna=False)[0:10].index

En el caso de datos númericos se puede sustituir con 'nan', cero, el promedio de los datos o un dato aleatorio de la población (un proceso similar a bootstrapping). Estas últimas opciones estan mas alla del alcance de este tutorial.

## 6.d Limpieza de texto

En este conjunto las columnas de texto son:
* 'Primer apellido'
* 'Segundo Apellido'
* 'Nombre(s)'
* 'Nombre completo'
* 'Institución_origen'
* 'Rdoc'

In [ ]:
col_str = ['Primer apellido', 'Segundo Apellido', 'Nombre(s)',  
           'Nombre completo', 'Institución_origen']

# Este comando obtiene para cada columna de interes los cinco valores mas comunes
# y los despliega para poder revisar los datos
for col in col_str:
    print(col)
    display( df[col].value_counts(dropna=False).index.tolist()[0:5] )

Dentro de estos datos tenemos varios tipos de valores que nos representan distintos tipos de datos faltantes.

Estos son:
* `nan`
* `'  '`
* 'Nombre de particular que se encuentra con vida, se clasifica como confidencial con fundamento en el artículo 116 de la LGTAIP.  '

Generalmente, podemos sustituir estos valores por `nan`, por un string vacio `''`o por un string especial cómo 'S/D' o 'Confidencial'.

En este caso vamos a sustituir por un string vacio: `''`

En primer lugar vamos a sustituir todos los `nan` usando el comando `.fillna()`.

In [ ]:
# esta operación solo se aplica y guarda sobre las columnas de texto
df[col_str] = df[col_str].fillna('')

for col in col_str:
    print(col)
    display( df[col].value_counts(dropna=False).index.tolist()[0:5] )

Ahora, vamos a remplazar los demas valores especiales:

* `'  '` --> `''`
* 'Nombre de particular que se encuentra con vida, se clasifica como confidencial con fundamento en el artículo 116 de la LGTAIP.  ' --> 'Confidencial'

En este caso crearemos un diccionario de remplazos y usaremos la función `.replace()`

In [ ]:
replace_str = {'  ':'',
               'Nombre de particular que se encuentra con vida, se clasifica como confidencial con fundamento en el artículo 116 de la LGTAIP.  ':'Confidencial'
              }

# remplazamos con un for loop columna por columna, en lugar de usar selección
# esto es solo por ejemplo

for col in col_str:
    print('Remplazando columna:', col)
    df[col] = df[col].replace(replace_str)
    # veamos los cinco valores mas comunes de la columna despúes de la sustitución
    display( df[col].value_counts(dropna=False).index.tolist()[0:5] )

Dependiendo del tipo de dato se pueden hacer diferentes operaciones. Por ejemplo, se puede pasar a mayusculas un texto pero no un número, se puede obtener el año de una fecha pero no de un texto

Por ejemplo, pensemos que queremos que las instituciones de origen no esten todas en mayusculas, sino solo la primera letra de la palabra.

Para hacer esto seleccionaremos la columna de interes, y luego aplicaremos la función `.str.title()`. 

**Nota**: existen varias funciones para mayusculas y minusculas como `.upper()`, `.lower()`, `.capitalize()` y `.title()`

In [ ]:
df['Institución_origen'].str.title()

Existen funciones que no se encuentran en pandas pero que pueden ser utiles. Por ejemplo, para quitar los acentos es posible con la función `unidecode()`, la cual es parte de la biblioteca `unidecode`.

In [ ]:
from unidecode import unidecode
unidecode('México')

Podemos aplicar esta función a toda la columna, en este caso usaremos apply, ya que no es una función default de pandas

In [ ]:
df['Institución_origen'].apply( unidecode )

Además, podemos aplicar varias funciones juntas.

In [ ]:
df['Institución_origen'].str.title().apply(unidecode)

Estas modificaciones son temporales. Para guardar los resultados es necesario guardar la columna o serie generada. Esto puede ser en una nueva columna o en la misma columna

In [ ]:
df['Institución_origen'] = df['Institución_origen'].str.title().apply(unidecode)
df.tail()

EL aplicar la función `unidecode()` a la columna puede genera un error si hay nans `AttributeError: 'float' object has no attribute 'encode'`. 
Esto se debe a que la columna incluye valores vacios, los cuales son de tipo `float`, por lo tanto la función `unidecode` falla, ya que los números no tienen acentos.

Dejo aqui una posible solución. Esta solución utiliza funciones lambda, las cuales son un tema avanzado más allá de este tutorial.

In [ ]:
df['Nombre(s)'].apply( lambda s: unidecode(s) if type(s)==str  else s )

Otra posible limpieza es quitar espacios al principio y final del texto, o cambiar tabuladores, saltos de página o múltiples espacios por uno solo. 
Este tipo de ocurrencias son dificiles de ver en el texto como humanos, sin embargo un espacio adicional al final de una palabra es suficiente para volverla un string diferente para la computadora. Por ejemplo, veamos este texto que incluye múltiples espaciós, tabuladores y saltos de linea.

In [ ]:
texto = "  Emanuel\tJurado     Sánchez \n "
print(texto)

La función `.strip()` quita los espacios adicionales del principio y final de un string.

In [ ]:
print(texto.strip())

Otra opción es combinar la función `.split()` y `.join()`.
La función `.split()` separa un texto en una lista, por default hace esto en los caracteres de espacio, pero se le pueden dar instrucciones de separar en caracteres específicos, por ejemplo en una coma `,`.
La función  `str.join()` une una lista con un caracter específico para generar un solo string.

En este caso estamos separando el texto en los espacios, para convertirlo en una lista de palabras. Despúes, esta lista se une con espacios.


In [ ]:
print(   ' '.join( texto.split() )   )

Es común que una limpieza requiera varios pasos. En ese caso podemos generar una función que incluya todos los pasos.

In [ ]:
def limpiar_texto(s):
    # checar el tipo
    if type(s)==str: # esto solo se ejecutará si es un string
        ### aqui agrega tus pasos de limpieza 
        s = unidecode(s) #quitar acento
        s = ' '.join( s.split() ) #quitar espacios extra
        s = s.title() #primera letra en mayuscula
    else: # esto se ejecutará si NO es un string
        s = s #devolveremos la entrada sin modificaciones, pero puedes cambiar esto
    return s
    
limpiar_texto(texto)

Ahora aplicaremos la función en las columnas de interes.

In [ ]:
for col in col_str:
    print('Limpieza de texto:', col)
    df[col] = df[col].apply( limpiar_texto )
df.tail()

## 6.e Limpieza de categóricos

En general se considera que un dato es categorico cuando hay pocas posibilidades definidas. Una aproximación rápida es que tenga menos de diez opciones de texto. Sin embargo esta no es una regla dura, por ejemplo el catálogo del INEGI de Estados, Municipios y Localidades tiene miles de opciones.


En este conjunto las columnas de texto son:
* 'Panteón_origen', 
* 'Estatus_FC', 
* 'Restos_tipo', 
* 'Sexo', 
* 'Conocido_Desconocido'

In [ ]:
col_cat = ['Panteón_origen', 'Estatus_FC', 'Restos_tipo', 'Sexo', 'Conocido_Desconocido']

# este es el mismo comando que usamos para ver col_str, pero cambiamos la lista que se usará
for col in col_cat:
    print(col)
    display( df[col].value_counts(dropna=False).index.tolist() )

Podemos convertir una columna de tipo `object` a `category` con la función `.astype("category")`. 

En este ejemplo caso reescribiremos la columna para poder realizar varias operaciones.

In [ ]:
for col in col_cat:
    df[col] = df[col].astype("category")

Podemos ver que el tipo a cambiado

In [ ]:
df.dtypes

Una ventaja de usar `category` es que permite definir un orden diferente al alfabético o numérico. Por ejemplo, definamos un orden especial para el tipo de restos. Por default las categorías se ordenan alfabeticamente, pero esto se puede modificar.

In [ ]:
from pandas.api.types import CategoricalDtype

orden_restos = ['Cadáver', 'Restos humanos', 'Restos cremados', 'Restos óseos', 'Feto', 'Miembros']
orden_restos = CategoricalDtype(categories=orden_restos, ordered=True)
df['Restos_tipo'] = df['Restos_tipo'].astype( orden_restos )
df['Restos_tipo'].tail()

Ahora si ordenamos el dataframe se usará el orden especial qué hemos definido

In [ ]:
df.sort_values(by='Restos_tipo')

### Creando columnas categóricas

Cuando se generan nuevas variables de datos a partir de las existentes se le llama "creación de características" o "ingeniería de características" (en inglés, feature engineering). Es una técnica comúnmente utilizada en el análisis de datos para mejorar la precisión de los modelos de machine learning o para encontrar patrones y tendencias en los datos. La idea es generar nuevas variables a partir de las ya existentes que puedan ser más relevantes o informativas para el problema en cuestión.

Veamos las instituciones de origen:

In [ ]:
df['Institución_origen'].value_counts()

Podemos ver que los cadaveres provienen de Instituciones Judiciales como en INCIFO y la PGR, de escuelas públicas y privadas. Entonces, generemos una nueva columna con el tipo de institución de origen.

Para esto primero crearemos un diccionario que nos permita mapear las instituciones a su tipo.

In [ ]:
cat_tipo_inst = {
        'Instituto De Ciencias Forenses - Tribunal Superior De Justicia De La Ciudad De Mexico': 'Institución judicial',
        'Procuraduria General De La Republica': 'Institución judicial',
        'Universidad Nacional Autonoma De Mexico - Facultad De Medicina': 'Escuela pública',
        'Instituto Politecnico Nacional - Escuela Nacional De Medicina Y Homeopatia': 'Escuela pública',
        'Centro Cultural Universitario Justo Sierra': 'Escuela privada',
        'Universidad Westhill - Facultad De Medicina': 'Escuela privada',
        'Universidad Anahuac - Facultad De Ciencias De La Salud': 'Escuela privada',
        'The American British Cowdray Medical Center La.P.': 'Escuela privada',
        'Secretaria De La Defensa Nacional - Escuela Militar De Medicina': 'Escuela privada',
        'Universidad Popular Autonoma Del Estado De Puebla': 'Escuela privada',
        'Universidad Tominaga Nakamoto S.C. Escuela De Medicina Ciencias Basicas': 'Escuela privada',
        'Escuela De Medicina Saint Luke': 'Escuela privada',
        'Institucion De Asistencia Privada - Escuela Libre De Homeopatia De Mexico': 'Escuela privada',
        }

A continuación usaremos map para generar una nueva columna usando el mapeo.

Guardaremos el resultado de esta operación en una columna nueva llamada 'Tipo_institución_origen'

**Nota** `.replace()` y `.map()` son funciones que toman un diccionario y lo utilizan para remplazar los valores. Sin embargo, `.replace()` solo sustituye los valores en el diccionario y si algo no se encuentra el valor en el dict deja la celda sin modificar. Por otro lado `.map()` cambia todos los valores en el diccionario y si no se encuentran el el dict lo sustituye por `nan`, por lo cual es útil para filtrar valores no validos.


In [ ]:
df['Tipo_institución_origen'] = df['Institución_origen'].map(cat_tipo_inst)
df['Tipo_institución_origen'].value_counts(dropna=False)

Ahora, hay que volver este dato de tipo categórico

In [ ]:
df['Tipo_institución_origen'] = df['Tipo_institución_origen'].astype("category")
df['Tipo_institución_origen'].head()

### Wibblywobbly o catálogos por similitud

A veces es necesario estandarizar los datos antes de volverlos una categoría.

Por ejemplo, estos serie de datos esta basada en datos llenados a mano sobre el sexo de pacientes.


In [ ]:
serie_sexo = pd.Series( [' Masculino', '-', '?', 'F', 'Femenina', 'Femenino', 'Femenino y Masculino', 'I?','Ilegible', 
                         'Indeterminable', 'Indeterminado', 'M', 'M F', 'MAsculino', 'Maculino', 'Masculino', 'Masculino.', 
                         'N', 'S/D', 'femenina', 'femenino', 'ilegible', 'indeterminado', 'masculino', 's/d', 'sin dato' ] ) 
serie_sexo

En primer lugar tenemos que determinar cuales opciones incluir en el catalogo. Además, es necesario tomar decisiones de registros dobles como 'M F'. 
En este caso nos gustaría reducir las opciones a 'Femenino', 'Masculino' e 'Indeterminado', donde 'Indeterminado' incluira todos los casos donde haya duda.

Para lograr la estandarización se generá un diccionario de equivalencias, el cual se usa con la función `.replace()` o `.map()`. 

Este diccionario se puede escribir a mano cómo se hizo arriba o inferir utilizando `wibblywobbly`. Esta biblioteca toma un catálogo y un conjunto de datos y regresa que tanto se parecen los textos.

In [ ]:
import wibblywobbly as ww

cat_sexo = ['Femenino', 'Masculino', 'Indeterminado']
ww.map_list_to_catalog(serie_sexo, cat_sexo, reject_value='Indeterminado')

También es posible obtener directamente un diccionario de sustitución.

La opción `reject_value` permite establecer un valor default si no se encuentra un texto lo suficientemente similar en el catálogo.

In [ ]:
replace_sexo = ww.map_list_to_catalog(serie_sexo, cat_sexo, output_format="dictionary", reject_value='Indeterminado')
replace_sexo

Este diccionario tiene errores, los cuales se pueden arreglar manualmente.

In [ ]:
replace_sexo['M'] = 'Masculino'
replace_sexo['N'] = 'Indeterminado'
replace_sexo['I?'] = 'Indeterminado'
replace_sexo['Femenino y Masculino'] = 'Indeterminado'

replace_sexo

Ahora podemos remplazar, estandarizar y contar.

In [ ]:
serie_sexo = serie_sexo.replace( replace_sexo )
serie_sexo

In [ ]:
serie_sexo.value_counts(dropna=False)

Revisen la documentación en: https://github.com/mar-esther23/WibblyWobbly

## 6.f Limpieza de números

Existen dos tipos de datos númericos, enteros y flotantes. Los dos se comportan de forma muy similar.

En este conjunto de datos la columna `Edad` contiene numeros, sin embargo su tipo es `object` ya que hay edades como "18 semanas" que corresponden a fetos y neonatos.
Además, cuando cargamos específicamos que el tipo era `str`. 


In [ ]:
df['Edad'].value_counts()

Por lo tanto, es necesario volver la edad a número. Esto se puede hacer con la función `.to_numeric()` o con `.astype()`.
Los datos que no se pueden convertir a número, por ejemplo '18 semanas', ya que generarán error. La función tiene varios parametros que nos pueden servir:
```
errors{‘ignore’, ‘raise’, ‘coerce’}, default ‘raise’
        If ‘raise’, then invalid parsing will raise an exception.
        If ‘coerce’, then invalid parsing will be set as NaN.
        If ‘ignore’, then invalid parsing will return the input.
```

Una primera aproximación es convertir obligar a que se convierta a número. Cómo los datos de semanas nos pueden ser utiles a futuro no vamos a rescribir la columna 'Edad', sino a generar una nueva columna 'Edad_int' con los resultados

In [ ]:
df['Edad_int'] = pd.to_numeric(df['Edad'], errors="coerce")
df['Edad_int'].unique()

Veamos las edades que no fueron convertidas por la operación:

In [ ]:
df.loc[ (df['Edad'].notna()) & (df['Edad_int'].isna()), ['Restos_tipo','Edad','Edad_int'] ].drop_duplicates()

La mayor parte de las edades en semanas son de fetos, mientras que las excepciones corresponden a rango. Dada la cantidad de faltantes ignoraremos estos datos. Sin embargo, es buena práctica revisar el resultado de las conversiones y donde fallo.

Ahora, veamos la distribución de edades:

In [ ]:
df['Edad_int'].describe()

Tenemos una persona de 101 años, esto podría ser un dato fuera de rango. 
En este caso vamos a poner un críterio para manejar problemas similares a futuro, si la persona tiene mas de cién años remplazaremos su edad por nan.
Para lograr esto seleccionaremos todas las celdas de personas de mas de cién años que están en la columna 'Edad_años' con _.loc[]_ y remplazaremos estos valores por NaN.
Esto debé de cambiar la edad máxima de nuestros datos.

In [ ]:
from numpy import nan

df.loc[ df['Edad_int']>=100,'Edad_int' ] = nan
df['Edad_int'].max()

Veamos las estadísticas de la columna 'Edad_años' usando `.describe()`, esta función es una alternativa rápida a un profile.

In [ ]:
df['Edad_int'].describe()

## 6.g Limpieza de fechas

Las fechas se encuentran en formato `datetime64`, lo cual incluye, fecha y hora.
Este formato sigue el patrón: `yyyy-mm-dd hh:mm:ss`

En este conjunto las columnas de texto son:
* 'Fecha_inhumación'
* 'Fecha_defunción'
* 'Marca_temporal'

In [ ]:
col_date = ['Fecha_inhumación', 'Fecha_defunción', 'Marca_temporal']

for col in col_date:
    print(col)
    display( df[col].value_counts(dropna=False).index.tolist()[0:5] )

Convertiremos las fechas a datetime con `to_datetime()` y veremos en que casos fracasa la función.
Guardaremos el resultado en una columna con el sufijo `_date`.

**Nota:** Para entender esta función es muy importante ver en donde se agrega el modificador que indica una nueva columna

In [ ]:
for col in col_date:
    print('Convirtiendo columna:' + col)
    df[col+'_date'] = pd.to_datetime(df[col], errors='coerce')
    display(df.loc[ (df[col].notna()) & (df[col+'_date'].isna()), [col, col+'_date'] ].drop_duplicates())

En este caso se puede ver que la conversión fallo solo en tres casos de la columna 'Fecha_defunción'. Cómo en estos casos falta información no es necesario hacer otra limpieza.

Usando este tipo de dato ahora es posible seleccionar por año, mes, día, hora y minuto usando los comandos:

* columna.dt.year
* columna.dt.month
* columna.dt.day
* columna.dt.hour
* columna.dt.minute
* columna.dt.dayofweek (numerico)
* columna.dt.weekday (nombre en ingles)
* columna.dt.dayofyear (numerico)
* columna.dt.weekofyear (numerico)

Agregaremos columnas para marcar el dia de la semana, del año y número de semana.

Es importante recordar que aplicaremos esto sobre las columnas procesadas con sufijo '_date'

In [ ]:
for col in col_date:
    col = col+'_date' #usemos columna modificada
    df[col+'_diasemana'] = df[col].dt.weekday
    df[col+'_diaaño'] = df[col].dt.dayofyear
    df[col+'_semanaaño'] = df[col].dt.weekofyear
df

Al agregar columnas es importante tomar en cuenta el tipo de dato. Por ejemplo, veamos las columnas de día de la semana. Estas asignan un número al día de la semana con Lunes=0 y Domingo=6.

In [ ]:
df['Marca_temporal_date_diasemana'].unique()

Editemos estas columnas para que tengan el día de la semana por su nombre y sean categóricos ordenados.

Lo primero es hacer un diccionario para sustituir los valores por el nombre del día de la semana.

In [ ]:
dic_semana = {0:'Lunes', 1:'Martes', 2:'Miercoles', 3:'Jueves',
              4:'Viernes', 5:'Sábado', 6:'Domingo'}

col_weekday = ['Fecha_inhumación_date_diasemana', 'Fecha_defunción_date_diasemana',
               'Marca_temporal_date_diasemana']

for col in col_weekday:
    df[col] = df[col].replace(dic_semana)
df

Ahora, generemos la categoría ordenada y transformemos las columnas a categórico.


In [ ]:
orden_semana = ['Lunes', 'Martes', 'Miercoles', 'Jueves',
                'Viernes', 'Sábado', 'Domingo']
orden_semana = CategoricalDtype(categories=orden_semana, ordered=True)

for col in col_weekday:
    df[col] = df[col].astype( orden_semana )
df.dtypes

## 6.h Ordenar y guardar datos

Antes de guardar los datos veamos las columnas, esto nos dará una idea de que hemos hecho.

In [ ]:
df.columns

Al agregar columnas estas se ponen al final, por lo que sería bueno ordenarlas. Además, las columnas de fecha originales se parecen mucho a las procesadas, excepto por el cambio de formato. Entonces, podríamos quitar las originales para hacer mas pequeña la tabla. Este también es un buen momento para arrepentirse, por ejemplo quitaremos las columnas de 'diaaño'.

Dependiendo del análisis podemos quitar columnas. Por ejemplo, la columna de `Marca_temporal` y sus columnas derivadas describen cuando se hizo la captura de la información del panteón al MFC. Esto es útil si nos interesa la información administrativa del Módulo, pero no si estamos interesados en los datos demográficos de los restos en Fosa común. Por lo tanto en este caso quitaremos estas columnas.


In [ ]:
df = df[['Panteón_origen', 'Estatus_FC', 
         'Fecha_inhumación_date', 'Fecha_inhumación_date_diasemana', 'Fecha_inhumación_date_semanaaño', 
         'Fecha_defunción_date', 'Fecha_defunción_date_diasemana', 'Fecha_defunción_date_semanaaño',
         'Restos_tipo', 'Sexo', 'Edad', 'Edad_int',  'Conocido_Desconocido',
         'Primer apellido', 'Segundo Apellido', 'Nombre(s)', 'Nombre completo',
         'Institución_origen', 'Tipo_institución_origen']]
df.tail()

Este es también un buen momento de cambiar los nombres de las columnas. 

**Nota** compara estos dos comandos para distinguir entre cambiar el orden de las columnas y en Nombre de estas.

In [ ]:
df.columns = ['Panteón_origen', 'Estatus_FC', 
              'Fecha_inhumación', 'Fecha_inhumación_diasemana', 'Fecha_inhumación_semanaaño', 
              'Fecha_defunción', 'Fecha_defunción_diasemana', 'Fecha_defunción_semanaaño',
              'Restos_tipo', 'Sexo', 'Edad', 'Edad_años',  'Conocido_Desconocido',
              'Primer_apellido', 'Segundo_Apellido', 'Nombres', 'Nombre_completo',
              'Institución_origen', 'Tipo_institución_origen']
df.tail()

Hagamos un nuevo profile para ver el comportamiento de los datos limpios

In [ ]:
#!pip install ydata_profiling

from ydata_profiling import ProfileReport

file_profile = "profiles/MFC_profile_clean.html"
prof = ProfileReport(df, minimal=True) 
prof.to_file(output_file=file_profile)

Es posible guardar los datos limpios que hemos obtenido de varias formas, podemos guardarlos como csv o excel usando los comandos `.to_csv()` y `.to_excel()`. Estos formatos tienen la ventaja de que son faciles de compartir. 

Nosotros guardaremos los datos limpios en la carpeta _data_clean_ como un csv, debido a que el tamaño de la base de datos es grande, puede ser complicado para manipularla en excel. Sin embargo, es posible abrir archivos csv usando excel.

Es muy importante no rescribir los datos originales y tratar de tener una carpeta para cada parte del proceso de análisis, para evitar perder información y poder reproducir confiablemente nuestros análisis.

In [ ]:
file_out = "data_clean/MFC_ActualizacionNov2022_clean.csv"
df.to_csv(file_out)

Una desventaja de guardar los archivos usando csv o excel, es que podemos perder el formato y los tipos de datos. Esto es importante sobretodo para tipos de datos como _datetime_. Una opción es guardar nuestros datos en un formato que sea facilmente interpretable para python, aunque este no se pueda trabajar con excel.

In [ ]:
from joblib import dump

file_out_pickle = "data_clean/MFC_ActualizacionNov2022_clean.pkl"

with open(file_out_pickle, 'wb') as f:
    dump(df, f)

## 6.i Resumen

**¡Gracias!**